In [ ]:
# ALL BASED ON PROOF OF 7.1 BENNETT ET AL !!!!!!!!!! USED FOR t(p^2)


# function to obtain potential d in t(p^d-1)=l
def get_d(l):
    eqn = l * ((l ** 2)-1) # theorem 1.1 by balakrishnan gives this eqn, d and odd prime divisors of this
    d = [i for i in eqn.prime_divisors() if (i % 2 != 0 and i != l)]
    return d

def congruence_checking(l,d_list): # lemma A.1 Goto, Ramanujan's prime power congruences
    # remaining d after congruence checking
    d_left = []
    for d in d_list:
        first,second,third,fourth = False,False,False,False
         # first congruence
        if (l % 12 in (0,1,d % 12)):
            first = True
            
        # second congruence
        if (d % 4) == 1:
            if (l % 5 in (0,1,d % 5)):
                second = True
        if (d % 4) == 3:
            if (l % 5 in (0,1,2,3,d % 5)):
                second = True

        # third congruence
        if (d % 6) == 1:
            if (l % 7 in (0,1,d % 7)):
                third = True
        if (d % 6) in (3,5):
            if (l % 7 in (0,1,2,4,d,(2*d) % 7,(4*d) % 7)):
                third = True
                
        # fourth congruence
        if (d % 6) == 1:
            if (l % 23 in (1,d % 23)):
                fourth = True
        if (d % 6) == 3:
            if (l % 23 in (0,1,d % 23)):
                fourth = True
        if (d % 6) == 5:
            if (l % 23 in (1,-1,d % 23)):
                fourth = True

        if all([first,second,third,fourth]): # check if all congruences are true for this potential d
            d_left.append(d)
    return d_left

primes = prime_range(252,1000)

data = dict()
for l in primes:
    d_list = get_d(l)
    
    filtered_d = congruence_checking(l, d_list)
    data[l] = list(filtered_d)

In [7]:
# ALL BASED ON PROOF OF 7.1 BENNETT ET AL !!!!!!!!!!

# get coefficient of fundamental unit

K.<sqrt601> = QuadraticField(601)

U = K.unit_group()
fundamental_unit = U.fundamental_units()[0]

# Factor a specific integer/rational within the number field
q = K.primes_above(23)[0]

for a in range(0, 11):
    F = q.residue_field()          # Create the residue field (GF(23))
    phi = F.reduction_map()        # Get the reduction homomorphism K -> GF(23)
    
    val_reduced = phi(2+sqrt601)    # Reduce sqrt(277) mod q
    unit_reduced = phi(fundamental_unit^a)
    if val_reduced == unit_reduced:
        print("a must be equal to: ", a)


a must be equal to:  4


In [4]:
# FUNDAMENTAL UNIT CHECKER
x = polygen(QQ)
K.<w> = NumberField(x^2 - 601)
U = K.unit_group()
fundamental_unit = U.fundamental_units()[0]
print(fundamental_unit)

5689030769845*w - 139468303679532


In [ ]:
# solve thue equations t(p^2) = l,
def generate_and_solve_thue(d, a, N=11, target=1):
    x = polygen(QQ)
    K.<w> = NumberField(x^2 - d)
    U = K.unit_group()
    fundamental_unit = U.fundamental_units()[0]
    unit_power = (fundamental_unit)^a
    
    c0 = QQ(unit_power[0])  # Rational coefficient
    c1 = QQ(unit_power[1])  # Coefficient of sqrt(d)
    print(c0, c1)
    
    """
    Constructs the Thue equation arising from the coefficient of sqrt(d) in
    (c0 + c1*sqrt(d)) * (U + V*sqrt(d))^N = Target, and solves it.
    
    Parameters:
    - d (int): Square-free integer for Q(sqrt(d)).
    - c0, c1 (QQ): Coefficients of the multiplier unit (c0 + c1*sqrt(d)).
    - N (int): Exponent / degree of the binary form.
    - target (int): Right-hand side integer of the Thue equation (default 1).
    
    Returns:
    - F: The bivariate polynomial F(U, V) representing the coefficient of sqrt(d).
    - solutions: List of integer pairs [U, V] satisfying F(U, V) = target.
    """
    # 1. Setup the polynomial ring for U, V over Rational Field
    R.<U, V> = PolynomialRing(QQ)
    
    # 2. Compute (U + V*sqrt(d))^N algebraically using Binomial Theorem
    #    (U + V*sqrt(d))^N = A(U,V) + B(U,V)*sqrt(d)
    A = R(0)
    B = R(0)
    
    for k in range(N + 1):
        term = binomial(N, k) * (U^(N - k)) * (V^k)
        if k % 2 == 0: # Even powers of sqrt(d) give rational integers d^(k/2)
            A += term * (d^(k // 2))
        else:          # Odd powers of sqrt(d) give d^((k-1)/2) * sqrt(d)
            B += term * (d^((k - 1) // 2))
            
    # 3. Multiply by (c0 + c1*sqrt(d)):
    #    (c0 + c1*sqrt(d)) * (A + B*sqrt(d)) = (c0*A + c1*B*d) + (c0*B + c1*A)*sqrt(d)
    #    We extract the coefficient of sqrt(d):
    F = c0 * B + c1 * A
    #reduce polynomial
    F_min = F.reduced_form()[0]
    print(F.reduced_form()[1:])

    # 4. Convert F(U, V) to univariate form f(x) = F(x, 1) for the PARI/GP solver
    f_univariate = F_min.subs({V: 1})
    
    # 5. Solve using PARI's thue solver
    pari_sols = gp.thue(f_univariate, target)
    solutions = pari_sols.sage()
    return F_min, solutions

print(generate_and_solve_thue(877,4))


27133539180403117523617 -916234515879791945688
([-385 -148]
[ -13   -5],)
